# Setup

#Clone - 27/03/2026
Fizemos um fork do git original para que possamos atualizar os scripts sem afetar o repositório original. No novo repositório realizamos a alteração .
O script MLOps -- Risco de Crédito - Regressao.ipynb passa a fazer o clone do novo repositório https://github.com/rafa-frederico/mlops2025.git

In [39]:
!git clone https://github.com/rafa-frederico/mlops2025.git

fatal: destination path 'mlops2025' already exists and is not an empty directory.


In [55]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

In [56]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Carrega os dados

In [57]:
mydf = pd.read_csv('./mlops2025/datasets/BaseDefault01.csv')
mydf

,nome,renda,idade,etnia,sexo,casapropria,outrasrendas,estadocivil,escolaridade,default
0,"Simon, Rodriguez",4472.190323,42.036031,0,0,1,0,0,3,0
1,"Daniel, Castro",4592.774312,48.230662,1,0,1,0,1,2,0
2,"Myhue, Lin",2486.538807,56.881709,0,1,0,0,0,0,1
3,"Destiny, Richardson-Pacheco",2852.340117,51.684021,1,1,0,0,0,2,1
4,"Brittany, Cohen-Wilson",4703.782812,50.729078,1,1,1,0,1,2,0
...,...,...,...,...,...,...,...,...,...,...
99995,"Darsean, Person",4716.190319,48.900391,0,0,1,1,1,2,0
99996,"Sulaimaan, al-Karim",4865.515077,60.924877,0,0,0,0,1,2,0
99997,"Tiffany, Gudavalli",2101.378948,23.531533,1,1,0,0,1,0,1
99998,"Althean, Farris",4408.155440,41.088311,0,0,0,1,1,1,0


# Treina o modelo

In [58]:
# Divide os dados em X e y, e posteriormente 80% em treino e 20% em teste
X = mydf[['renda', 'idade', 'etnia', 'sexo', 'casapropria', 'outrasrendas', 'estadocivil', 'escolaridade']]
y = mydf['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [59]:
# Train the logistic regression model
logreg = LogisticRegression(solver='liblinear')
logreg.fit(X_train, y_train)

LogisticRegression(solver='liblinear')

In [60]:
# Faz as previsões da base de teste
y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)

display(y_pred)
display(y_proba)

array([0, 0, 0, ..., 1, 0, 0])

array([[0.81383481, 0.18616519],
       [0.96081751, 0.03918249],
       [0.80814411, 0.19185589],
       ...,
       [0.26942677, 0.73057323],
       [0.98082765, 0.01917235],
       [0.7690274 , 0.2309726 ]])

In [61]:
# Avalia o modelo
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.89      0.89     11036
           1       0.87      0.85      0.86      8964

    accuracy                           0.87     20000
   macro avg       0.87      0.87      0.87     20000
weighted avg       0.87      0.87      0.87     20000

[[9867 1169]
 [1346 7618]]


In [62]:
# Identificação dos coeficientes e Intercepto
print("Coeficientes:", logreg.coef_)
print("Intercepto:", logreg.intercept_)

Coeficientes: [[-0.00234534 -0.0151067   1.38588731  1.36832242 -2.16535348 -0.8794301
  -0.56239811 -0.58691305]]
Intercepto: [9.71123274]


# Função de Predição

In [63]:
# Preencha essa função com os coeficientes corretos e com o valor correto de intercepto
def predict_default(renda, idade, etnia, sexo, casapropria, outrasrendas, estadocivil, escolaridade):
  entrada = [renda, idade, etnia, sexo, casapropria, outrasrendas, estadocivil, escolaridade]

#alterado com os valores de treinamento
  coeficientes = [-0.00234534, -0.0151067, 1.38588731, 1.36832242, -2.16535348, -0.8794301, -0.56239811, -0.58691305]
  intercepto = 9.71123274

  z = intercepto + sum(entrada[i] * coeficientes[i] for i in range(len(entrada)))

  # Calcula a probabilidade usando a função sigmóide e classifica de acordo com a probabilidade
  probabilidade = 1 / (1 + math.exp(-z))
  classificacao = 1 if probabilidade >= 0.5 else 0

  return {"probabilidade": probabilidade, "classificacao": classificacao}

## Testes da função de predição

Teste 01: Dados fixos

In [64]:
predict_default(3000, 30, 1, 0, 1, 0, 1, 0)

{'probabilidade': 0.7068527774965273, 'classificacao': 1}

Teste 02 : Se quiser, faça novas chamadas com valores de um cliente aleatório

In [65]:
cliente_aleatorio = X_test.sample(1)
display( cliente_aleatorio )
display( y_test[cliente_aleatorio.index] )

,renda,idade,etnia,sexo,casapropria,outrasrendas,estadocivil,escolaridade
21588,3589.906315,26.184669,1,0,0,0,0,1


,default
21588,1


---------

# Extra: Cria modelos com Random Forest

Extra: Nesta parte extra, avalie o arquivo `train_model.py` para entender como funciona um script já pronto para um modelo pré-definido e que precisa de seus coeficientes/pesos atualizados.

In [66]:
import mlops2025.train_model

Novo método train_models - adicionado novos parâmetros para permitir treinar o modelo com um número maior de florestas e altura.

In [67]:
#@markdown Informe a quantidade de florestas:
florestas = 20 #@param {type:"integer"}

#@markdown Informe a altura para modelo 01 - classificador
altura_01 = 10 #@param {type:"integer"}

#@markdown Informe a altura para modelo 02 - regressão
altura_02 = 8 #@param {type:"integer"}

In [68]:
mlops2025.train_model.train_models(mydf, florestas, altura_01, altura_02)

Modelo 01 (classificador), criado com acurácia de: [0.8808]
Modelo 02 (Regressor), criado com acurácia de: [0.6162978666116059]
Modelo 01 (classificador) salvo com sucesso.
Modelo 02 (regressor) salvo com sucesso.


In [69]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
